# Лекция: Графическое решение задач линейного программирования в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 15** (адаптация с языка R на Python)

## Краткая теория

**Линейное программирование (ЛП):** оптимизация линейной целевой функции при линейных ограничениях.

Для **двух переменных** задачу можно решить **графически**:
1. Построить область допустимых решений (многоугольник)
2. Нарисовать линии уровня целевой функции
3. Сдвигать линию уровня в направлении градиента до крайней вершины

В Python: **matplotlib** + **scipy.optimize.linprog**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from scipy.optimize import linprog

plt.rcParams['figure.figsize'] = (9, 7)
print("Библиотеки загружены")


---
## 1. Простейшая задача: прямоугольная область

max/min c1*x1 + c2*x2 при a1 <= x1 <= b1, a2 <= x2 <= b2.

В R-примере: c=(1,3), область [2,6] x [1,3].


In [ ]:
c1, c2 = 1, 3
a1, b1 = 2, 6
a2, b2 = 1, 3
eps = 1

fig, ax = plt.subplots()
ax.set_xlim(0, b1 + eps)
ax.set_ylim(0, b2 + eps)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Графическая интерпретация ЗЛП (прямоугольник)")

rect = Rectangle((a1, a2), b1 - a1, b2 - a2,
                 facecolor="salmon", edgecolor="darkred", alpha=0.6, label="допустимая область")
ax.add_patch(rect)

verts = [(a1, a2), (a1, b2), (b1, b2), (b1, a2)]
labels = ["A", "B", "C", "D"]
for (x, y), lab in zip(verts, labels):
    ax.plot(x, y, "bo", markersize=8)
    ax.text(x + 0.15, y + 0.1, lab, color="blue", fontsize=12)

ax.plot(0, 0, "ko", markersize=6)
ax.text(0.1, 0.15, "O", fontsize=11)

ax.annotate("", xy=(c1, c2), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax.text(c1 + 0.1, c2 + 0.1, "grad f", color="blue")

def draw_level(C, ax, style="-"):
    xs = np.linspace(0, b1 + eps, 200)
    ys = (C - c1 * xs) / c2
    ax.plot(xs, ys, style, color="steelblue", alpha=0.7)
    ax.text(xs[-1] * 0.85, (C - c1 * xs[-1] * 0.85) / c2 + 0.05,
            f"{C:.0f}", color="steelblue", fontsize=9)

Cmin = c1 * a1 + c2 * a2
Cmax = c1 * b1 + c2 * b2
draw_level(Cmin, ax, style="--")
draw_level(Cmax, ax, style="--")
for C in np.linspace(0, c1 * (b1 + eps) + c2 * (b2 + eps), 12):
    draw_level(C, ax)

ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"f(A)={c1*a1+c2*a2}, f(B)={c1*a1+c2*b2}, f(C)={c1*b1+c2*b2}, f(D)={c1*b1+c2*a2}")
print(f"Cmin = {Cmin}, Cmax = {Cmax}")


---
## 2. ЗЛП с линейными ограничениями-неравенствами

max c1*x1 + c2*x2 при A*x <= b, x >= 0.

Пример: максимум около 5.33 в точке около (1.67, 0.67).


In [ ]:
c1, c2 = 2, 2
a11, a12, b1 = 1, 2, 3
a21, a22, b2 = 2, 1, 4

A = np.array([[a11, a12], [a21, a22]], dtype=float)
rhs = np.array([b1, b2], dtype=float)
xP, yP = np.linalg.solve(A, rhs)
print(f"Пересечение ограничений: ({xP:.3f}, {yP:.3f})")

vertices = [
    (0, 0),
    (min(b1 / a11, b2 / a21), 0),
    (xP, yP),
    (0, min(b1 / a12, b2 / a22)),
]

def feasible(pt):
    x, y = pt
    return (a11*x + a12*y <= b1 + 1e-9 and
            a21*x + a22*y <= b2 + 1e-9 and
            x >= -1e-9 and y >= -1e-9)

vertices = [v for v in vertices if feasible(v)]
print("Вершины:", [(round(x, 3), round(y, 3)) for x, y in vertices])


In [ ]:
fig, ax = plt.subplots()
Xmax = max(b1 / a11, b2 / a21) + 1
Ymax = max(b1 / a12, b2 / a22) + 1
ax.set_xlim(-0.3, Xmax)
ax.set_ylim(-0.3, Ymax)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Графическая интерпретация ЗЛП")
ax.axhline(0, color="k", lw=1)
ax.axvline(0, color="k", lw=1)

xs = np.linspace(0, Xmax, 200)
ax.plot(xs, (b1 - a11 * xs) / a12, "g-", label=f"{a11}*x1 + {a12}*x2 = {b1}")
ax.plot(xs, (b2 - a21 * xs) / a22, "m-", label=f"{a21}*x1 + {a22}*x2 = {b2}")

poly = Polygon(vertices, closed=True, facecolor="salmon",
               edgecolor="darkred", lw=2, alpha=0.5, label="допустимая область")
ax.add_patch(poly)

labs = ["A", "B", "C", "D"]
for (x, y), lab in zip(vertices, labs):
    ax.plot(x, y, "bo", ms=8)
    ax.text(x + 0.05, y + 0.05, lab, color="blue", fontsize=12)

ax.annotate("", xy=(c1 * 0.8, c2 * 0.8), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax.text(c1 * 0.8 + 0.05, c2 * 0.8 + 0.05, "grad f", color="blue")

vals = [c1 * x + c2 * y for x, y in vertices]
Cmin, Cmax = min(vals), max(vals)
print("f в вершинах:", [round(v, 3) for v in vals])
print(f"Cmin={Cmin:.3f}, Cmax={Cmax:.3f}")
imax = int(np.argmax(vals))
print(f"Максимум в вершине {labs[imax]} = {vertices[imax]}")

def draw_level(C, ax):
    xs = np.linspace(-0.5, Xmax, 200)
    ys = (C - c1 * xs) / c2
    ax.plot(xs, ys, color="steelblue", alpha=0.5, lw=1)

draw_level(Cmin, ax)
draw_level(Cmax, ax)
for C in np.linspace(0, Cmax + 2, 8):
    draw_level(C, ax)

ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
res = linprog(
    c=[-c1, -c2],
    A_ub=[[a11, a12], [a21, a22]],
    b_ub=[b1, b2],
    bounds=[(0, None), (0, None)],
    method="highs",
)
print("linprog status:", res.message)
print(f"x* = ({res.x[0]:.4f}, {res.x[1]:.4f})")
print(f"max f = {-res.fun:.4f}")


---
## 3. Задача планирования производства

Спланировать объёмы производства двух товаров x1, x2 так, чтобы максимизировать доход.

Типовая таблица (подставьте свои числа из задания):

| Ресурс | Товар 1 | Товар 2 | Запас |
|--------|---------|---------|-------|
| Материал | 2 | 1 | 10 |
| Труд | 1 | 2 | 8 |
| Цена (доход) | 5 | 4 | — |

max 5*x1 + 4*x2 при 2*x1 + x2 <= 10, x1 + 2*x2 <= 8, x >= 0.


In [ ]:
c = np.array([5.0, 4.0])
A_ub = np.array([[2.0, 1.0], [1.0, 2.0]])
b_ub = np.array([10.0, 8.0])

res = linprog(c=-c, A_ub=A_ub, b_ub=b_ub,
              bounds=[(0, None), (0, None)], method="highs")
print("Оптимальный план:", res.x.round(4))
print("Максимальный доход:", (-res.fun).round(4))

try:
    xp = np.linalg.solve(A_ub, b_ub)
except np.linalg.LinAlgError:
    xp = (0, 0)
verts = [(0, 0),
         (b_ub[0] / A_ub[0, 0], 0) if A_ub[0, 0] else (0, 0),
         tuple(xp),
         (0, b_ub[1] / A_ub[1, 1]) if A_ub[1, 1] else (0, 0)]

def feas(pt):
    x = np.array(pt)
    return np.all(A_ub @ x <= b_ub + 1e-8) and np.all(x >= -1e-8)

verts = [v for v in verts if feas(v)]
verts = sorted(verts, key=lambda p: np.arctan2(p[1], p[0]))

fig, ax = plt.subplots()
xmax = max(v[0] for v in verts) + 2
ymax = max(v[1] for v in verts) + 2
ax.set_xlim(-0.5, xmax)
ax.set_ylim(-0.5, ymax)
ax.axhline(0, color="k"); ax.axvline(0, color="k")

xs = np.linspace(0, xmax, 200)
for i in range(A_ub.shape[0]):
    if abs(A_ub[i, 1]) > 1e-12:
        ax.plot(xs, (b_ub[i] - A_ub[i, 0] * xs) / A_ub[i, 1], label=f"огр. {i+1}")

poly = Polygon(verts, closed=True, facecolor="lightgreen",
               edgecolor="darkgreen", lw=2, alpha=0.5)
ax.add_patch(poly)

for v in verts:
    ax.plot(*v, "ko", ms=7)
    fval = c @ np.array(v)
    ax.text(v[0] + 0.1, v[1] + 0.1, f"({v[0]:.1f},{v[1]:.1f})\nf={fval:.1f}", fontsize=9)

opt = res.x
ax.plot(opt[0], opt[1], "r*", ms=18, label=f"optimum f={-res.fun:.2f}")
ax.annotate("", xy=c / np.linalg.norm(c) * 2, xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="red", lw=2))
ax.set_xlabel("x1 (товар 1)")
ax.set_ylabel("x2 (товар 2)")
ax.set_title("Планирование производства: max доход")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Как решить свою задачу

1. Запишите целевую функцию c1*x1 + c2*x2 -> max (или min).
2. Выпишите ограничения A*x <= b, x >= 0.
3. Найдите вершины допустимого многоугольника.
4. Вычислите f в каждой вершине — экстремум в одной из них.
5. Проверьте численно через linprog.
6. На графике: область + линии уровня + градиент + оптимум.

---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `plot` / `rect` / `polygon` | `matplotlib` patches |
| `abline`, `arrows` | `ax.plot`, `ax.annotate` |
| `text`, `points` | `ax.text`, `ax.plot(..., 'o')` |
| (нет встроенного LP) | `scipy.optimize.linprog` |

---
## Рекомендации

1. Для 2 переменных графический метод обязателен к пониманию.
2. При n > 2 используйте только linprog.
3. Если в задании 3 есть своя таблица — подставьте коэффициенты.
4. method='highs' — современный солвер.

**Удачи с выполнением Задания 15!**
